# DLIS Unit II: CIFAR-10 Model Comparison
**Dev Kumar Rajak | PES1PG25CA053 | PES University**  
Deep Learning and Intelligent Systems (UQ25CA754AB3), September 2026

Compare A1 (Dense), A2 (CNN with prescribed grid CV), and A3 (frozen ImageNet ResNet50).
This notebook follows the supplied five-page classroom handout. It is **unexecuted**:
all results must come from running it with the competition data. No accuracy is assumed.
The supplied starter notebook was reviewed. Dataset loading, class mapping, stratified splits
and submission ID alignment were checked against the uploaded competition package;
TensorFlow training has not been run here.

## Before running
1. Import this `.ipynb` into a Kaggle notebook for this competition.
2. Attach the supplied `train.npz`, `test.npz`, `sample_submission.csv`, and `class_names.csv`.
3. Enable a GPU and Internet (Internet is needed for ImageNet weights unless cached).
4. Run cells in order. Automatic path detection works if exactly one `train.npz` is attached;
   otherwise set `DATA_DIR` below to its parent folder from the Data panel.
5. Review the generated evidence and complete the discussion cell before exporting the two PDFs.

Only supplied competition images and labels are used. Do not download public CIFAR-10 copies
or try to recover hidden labels. The prescribed ImageNet weights are used only for A3.


In [ ]:
import gc
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from IPython.display import display, Markdown
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

SEED = 42
BATCH_SIZE = 64
OUT = Path('/kaggle/working')
OUT.mkdir(parents=True, exist_ok=True)
tf.keras.utils.set_random_seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow:', tf.__version__)
print('All physical devices:', tf.config.list_physical_devices())
print('GPU devices:', gpus)
if not gpus:
    raise RuntimeError('Enable a GPU in Kaggle notebook settings, restart, and rerun.')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as exc:
        print('Memory-growth setting:', exc)

print('Seed:', SEED, '| batch size:', BATCH_SIZE)
print('GPU operations may still have small numerical differences across environments.')


## 1. Load and inspect the supplied data
Keep raw RGB values on the 0-255 scale. A1/A2 rescale inside their models.
A3 resizes and applies ResNet-specific preprocessing inside its model.


In [ ]:
# If automatic detection finds several datasets, replace None with the folder path:
# DATA_DIR = Path('/kaggle/input/your-competition-slug')
DATA_DIR = None

if DATA_DIR is None:
    candidates = sorted(Path('/kaggle/input').rglob('train.npz'))
    if len(candidates) != 1:
        raise ValueError(f'Set DATA_DIR explicitly. Found train.npz files: {candidates}')
    DATA_DIR = candidates[0].parent
else:
    DATA_DIR = Path(DATA_DIR)
print('Data folder:', DATA_DIR)

def data_file(filename):
    # Accept underscores or spaces in CSV filenames, but require an unambiguous match.
    normalized = filename.lower().replace('_', '').replace(' ', '')
    matches = [p for p in DATA_DIR.iterdir() if p.is_file()
               and p.name.lower().replace('_', '').replace(' ', '') == normalized]
    if len(matches) != 1:
        raise FileNotFoundError(f'Expected one {filename} in {DATA_DIR}; found {matches}')
    return matches[0]

with np.load(data_file('train.npz'), allow_pickle=False) as train:
    print('train.npz keys:', train.files)
    x_all, y_all = train['images'], train['labels'].reshape(-1)
with np.load(data_file('test.npz'), allow_pickle=False) as test:
    print('test.npz keys:', test.files)
    x_test, test_ids = test['images'], test['ids'].reshape(-1)

sample_submission = pd.read_csv(data_file('sample_submission.csv'))
class_table = pd.read_csv(data_file('class_names.csv'))
display(class_table)
display(sample_submission.head())

assert x_all.shape == (15000, 32, 32, 3), x_all.shape
assert x_test.shape == (3000, 32, 32, 3), x_test.shape
assert len(y_all) == len(x_all) and len(test_ids) == len(x_test)
assert np.issubdtype(y_all.dtype, np.integer), 'Labels must be sparse integers.'
assert set(np.unique(y_all)) == set(range(10))
y_all = y_all.astype(np.int32)

for name, arr in [('training images', x_all), ('test images', x_test)]:
    print(name, '| shape:', arr.shape, '| dtype:', arr.dtype,
          '| pixel range:', (arr.min(), arr.max()))
    assert np.isfinite(arr).all() and 0 <= arr.min() and arr.max() <= 255
    assert arr.max() > 1, 'Expected raw 0-255 images, not already normalized images.'
print('Labels:', y_all.shape, y_all.dtype)
assert list(sample_submission.columns) == ['id', 'label']
assert len(sample_submission) == len(x_test)
assert sample_submission['id'].is_unique and sample_submission['id'].notna().all()
assert pd.Index(test_ids).is_unique and not pd.isna(test_ids).any()
assert set(sample_submission['id']) == set(test_ids), 'Test IDs differ from the sample.'

# These column names were verified in the supplied competition package.
assert {'label', 'class_name'}.issubset(class_table.columns)
assert len(class_table) == 10 and class_table['label'].is_unique
assert set(class_table['label']) == set(range(10))
assert class_table['class_name'].notna().all()
class_map = dict(zip(class_table['label'].astype(int), class_table['class_name'].astype(str)))
class_names = [class_map[i] for i in range(10)]
print('Verified class mapping:', class_map)


In [ ]:
counts = pd.Series(y_all).value_counts().sort_index()
distribution = pd.DataFrame({'label': counts.index, 'class': class_names, 'count': counts.values})
display(distribution)
distribution.plot.bar(x='class', y='count', legend=False, figsize=(10, 3), rot=35)
plt.title('Class distribution: supplied labelled data')
plt.ylabel('Images')
plt.tight_layout()
plt.show()

rng = np.random.default_rng(SEED)
sample_indices = rng.choice(len(x_all), size=12, replace=False)
fig, axes = plt.subplots(3, 4, figsize=(10, 7))
for ax, i in zip(axes.flat, sample_indices):
    ax.imshow(x_all[i].astype(np.uint8))
    ax.set_title(f'{y_all[i]}: {class_names[y_all[i]]}')
    ax.axis('off')
plt.tight_layout()
plt.show()


Each image has one **integer class label from 0 to 9**, not a one-hot vector.
Sparse categorical cross-entropy accepts these integer targets and compares them with
the model's ten softmax probabilities. ReLU introduces nonlinearity in the hidden layers;
softmax gives probabilities summing to one. Adam updates weights using adaptive learning rates.

## 2. Fixed final split and tuning subset
Use seed 42 and the same stratified 80:20 split for every final model. The 4,000-image
stratified tuning subset comes **only from the 12,000 training images**. A2 CV never sees
the 3,000 final validation images. The hidden test images are used only after model selection.


In [ ]:
train_indices, val_indices = train_test_split(
    np.arange(len(y_all)), test_size=0.20, stratify=y_all, random_state=SEED)
x_train, y_train = x_all[train_indices], y_all[train_indices]
x_val, y_val = x_all[val_indices], y_all[val_indices]
tune_indices, _ = train_test_split(
    train_indices, train_size=4000, stratify=y_all[train_indices], random_state=SEED)
x_tune, y_tune = x_all[tune_indices], y_all[tune_indices]
assert set(train_indices).isdisjoint(val_indices)
assert set(tune_indices).issubset(set(train_indices))
assert len(x_tune) == 4000
np.savez(OUT / 'split_indices.npz', train=train_indices, validation=val_indices, tuning=tune_indices)
display(pd.DataFrame({
    'class': class_names,
    'final_train': np.bincount(y_train, minlength=10),
    'final_validation': np.bincount(y_val, minlength=10),
    'tuning_subset': np.bincount(y_tune, minlength=10)}))
print('Final train:', x_train.shape, '| validation:', x_val.shape, '| tuning:', x_tune.shape)


## 3. Shared training and evidence helpers
For each final model, early stopping monitors validation loss with patience 2 and restores
the best weights. Timing covers only `model.fit`, including its validation passes and callbacks;
it excludes model construction, ImageNet download, subsequent evaluation, saving and prediction.

**Best validation accuracy** is the highest epoch accuracy. **Final validation accuracy** is
measured after restoring the lowest-validation-loss weights, so it need not equal the peak accuracy.


In [ ]:
L = tf.keras.layers
records, histories, val_probabilities = {}, {}, {}

def reset_session():
    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(SEED)

def compile_model(model, learning_rate=1e-3):
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def parameter_counts(model):
    return model.count_params(), int(sum(np.prod(v.shape) for v in model.trainable_weights))

def plot_history(name, history):
    epochs = np.arange(1, len(history['loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    for ax, metric in zip(axes, ['loss', 'accuracy']):
        ax.plot(epochs, history[metric], 'o-', label='Training')
        ax.plot(epochs, history['val_' + metric], 'o-', label='Validation')
        ax.set(xlabel='Epoch', ylabel=metric, title=f'{name}: {metric}', xticks=epochs)
        ax.legend()
        ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(OUT / f'{name}_curves.png', dpi=160, bbox_inches='tight')
    plt.show()

def train_final(name, model, max_epochs):
    model.summary()
    total, trainable = parameter_counts(model)
    print('Total parameters:', total, '| trainable:', trainable)
    stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', mode='min', patience=2, restore_best_weights=True)
    start = time.perf_counter()
    history = model.fit(x_train, y_train, validation_data=(x_val, y_val),
                        epochs=max_epochs, batch_size=BATCH_SIZE, shuffle=True,
                        callbacks=[stop], verbose=2)
    fit_seconds = time.perf_counter() - start
    h = history.history
    val_loss, val_acc = model.evaluate(x_val, y_val, batch_size=BATCH_SIZE, verbose=0)
    train_loss, train_acc = model.evaluate(x_train, y_train, batch_size=BATCH_SIZE, verbose=0)
    probabilities = model.predict(x_val, batch_size=BATCH_SIZE, verbose=0)
    assert np.isfinite(probabilities).all()
    records[name] = dict(
        model=name, total_params=total, trainable_params=trainable,
        time_s=float(fit_seconds), epochs=len(h['loss']),
        best_val_loss=float(min(h['val_loss'])),
        best_val_acc=float(max(h['val_accuracy'])),
        restored_epoch=int(np.argmin(h['val_loss']) + 1),
        final_val_loss=float(val_loss), final_val_acc=float(val_acc),
        final_train_loss=float(train_loss), final_train_acc=float(train_acc))
    histories[name] = {k: [float(v) for v in values] for k, values in h.items()}
    val_probabilities[name] = probabilities
    model.save_weights(str(OUT / f'{name}.weights.h5'))
    pd.DataFrame(h).to_csv(OUT / f'{name}_history.csv', index=False)
    display(pd.DataFrame([records[name]]).round(4))
    plot_history(name, h)


## 4. A1: Dense baseline
`Rescaling -> Flatten -> Dense(256, ReLU) -> Dense(128, ReLU) -> Dropout(0.2) -> Dense(10, softmax)`.
Adam learning rate 0.001, maximum 5 epochs.

Flatten produces 3,072 inputs. The first Dense layer alone has
`3072 * 256 + 256 = 786,688` parameters. Flatten retains the pixel values but the Dense
architecture has no explicit local-neighbourhood or shared-filter structure.


In [ ]:
def build_a1():
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(32, 32, 3)),
        L.Rescaling(1.0 / 255),
        L.Flatten(),
        L.Dense(256, activation='relu'),
        L.Dense(128, activation='relu'),
        L.Dropout(0.2),
        L.Dense(10, activation='softmax')], name='A1_Dense')
    return compile_model(model)

reset_session()
model = build_a1()
train_final('A1', model, max_epochs=5)
del model


## 5. A2: prescribed CNN grid search
Three 3x3 Conv2D layers, each with ReLU and 2x2 max pooling, followed by Flatten,
Dense(128, ReLU), dropout and Dense(10, softmax). Convolutions use `padding='same'`:
the pooling stages reduce the spatial size from 32 to 16 to 8 to 4.

Exactly four configurations, two stratified folds, and exactly two epochs per fold.
Each fold starts with a fresh model and optimizer. There is no early stopping during CV.
Select the largest mean best-fold validation accuracy; exact ties prefer fewer parameters,
then smaller summed fit time. The reported standard deviation uses `ddof=0`.


In [ ]:
GRID = [
    dict(config='G1', filters=(16, 32, 64), learning_rate=1e-3, dropout=0.2),
    dict(config='G2', filters=(16, 32, 64), learning_rate=3e-4, dropout=0.3),
    dict(config='G3', filters=(32, 64, 128), learning_rate=1e-3, dropout=0.2),
    dict(config='G4', filters=(32, 64, 128), learning_rate=3e-4, dropout=0.3)]
display(pd.DataFrame(GRID))

def build_a2(config):
    layers = [tf.keras.Input(shape=(32, 32, 3)), L.Rescaling(1.0 / 255)]
    for filters in config['filters']:
        layers += [L.Conv2D(filters, 3, padding='same', activation='relu'),
                   L.MaxPooling2D(pool_size=2)]
    layers += [L.Flatten(), L.Dense(128, activation='relu'),
               L.Dropout(config['dropout']), L.Dense(10, activation='softmax')]
    return compile_model(tf.keras.Sequential(layers, name='A2_CNN'), config['learning_rate'])

cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=SEED)
folds = list(cv.split(x_tune, y_tune))
fold_records = []
for config in GRID:
    for fold, (fit_idx, valid_idx) in enumerate(folds, start=1):
        reset_session()
        model = build_a2(config)
        total, _ = parameter_counts(model)
        print(f"{config['config']} | fold {fold}/2 | {total:,} parameters")
        start = time.perf_counter()
        history = model.fit(
            x_tune[fit_idx], y_tune[fit_idx],
            validation_data=(x_tune[valid_idx], y_tune[valid_idx]),
            epochs=2, batch_size=BATCH_SIZE, shuffle=True, verbose=2)
        elapsed = time.perf_counter() - start
        assert len(history.history['loss']) == 2
        fold_records.append(dict(
            config=config['config'], fold=fold, total_params=total, time_s=elapsed,
            best_fold_val_acc=float(max(history.history['val_accuracy']))))
        del model, history

fold_results = pd.DataFrame(fold_records)
grid_results = fold_results.groupby('config', as_index=False).agg(
    mean_val_acc=('best_fold_val_acc', 'mean'),
    std_val_acc=('best_fold_val_acc', lambda x: np.std(x, ddof=0)),
    total_params=('total_params', 'first'), time_s=('time_s', 'sum'))
grid_results = pd.DataFrame(GRID).merge(grid_results, on='config')
grid_results = grid_results.sort_values(
    ['mean_val_acc', 'total_params', 'time_s'], ascending=[False, True, True]).reset_index(drop=True)
display(fold_results.round(5))
display(grid_results.round(5))
fold_results.to_csv(OUT / 'a2_fold_results.csv', index=False)
grid_results.to_csv(OUT / 'a2_grid_results.csv', index=False)
best_config = next(c.copy() for c in GRID if c['config'] == grid_results.iloc[0]['config'])
print('Selected A2 configuration:', best_config)
print('Total CV fit time (s):', round(fold_results['time_s'].sum(), 2))


## 6. A2: train the selected configuration from scratch
Use the fixed 12,000 training images and 3,000 validation images, maximum 6 epochs.
No weights are carried over from cross-validation. Final fit time is reported separately
from the extra tuning cost.


In [ ]:
reset_session()
model = build_a2(best_config)
train_final('A2', model, max_epochs=6)
del model


## 7. A3: frozen ImageNet ResNet50
Resize raw RGB images from 32x32 to 96x96, then apply `resnet.preprocess_input`.
This converts RGB to BGR and subtracts ImageNet channel means, without division by 255.
Never combine it with `Rescaling(1/255)` here.

ResNet50 has `include_top=False`, `pooling='avg'`, `weights='imagenet'` and a frozen base.
Calling the base with `training=False` keeps its batch-normalization layers in inference mode.
Only Dropout(0.3) and a 10-unit softmax Dense head are attached. Adam uses 0.001,
maximum 3 epochs. There is no optional fine-tuning in this required comparison.

The Functional API makes the explicit `training=False` call clear; A1 and A2 use Sequential.
The pretrained model, larger input and preprocessing cost make this a practical comparison,
not an equal-compute or equal-prior-knowledge experiment.


In [ ]:
def build_a3(weights='imagenet'):
    base = tf.keras.applications.ResNet50(
        include_top=False, weights=weights, input_shape=(96, 96, 3), pooling='avg')
    base.trainable = False
    inputs = tf.keras.Input(shape=(32, 32, 3), name='raw_rgb')
    x = L.Resizing(96, 96, interpolation='bilinear', name='resize_96')(inputs)
    x = L.Lambda(tf.keras.applications.resnet.preprocess_input,
                 name='resnet_preprocessing')(x)
    x = base(x, training=False)
    x = L.Dropout(0.3)(x)
    outputs = L.Dense(10, activation='softmax')(x)
    model = tf.keras.Model(inputs, outputs, name='A3_Frozen_ResNet50')
    assert not base.trainable and len(base.trainable_weights) == 0
    return compile_model(model)

reset_session()
model = build_a3()
assert parameter_counts(model)[1] == (2048 + 1) * 10
train_final('A3', model, max_epochs=3)
del model


## 8. Comparison and validation-only selection
Choose by **final validation accuracy after best-loss weight restoration**. If tied,
prefer lower final validation loss, then fewer trainable parameters. The highest epoch
accuracy is also reported, but it may belong to weights that were not retained.
The hidden test set and leaderboard are not used for this choice.


In [ ]:
comparison = pd.DataFrame(list(records.values()))
comparison['train_val_gap'] = comparison['final_train_acc'] - comparison['final_val_acc']
main_columns = ['model', 'total_params', 'trainable_params', 'time_s', 'epochs',
                'best_val_loss', 'best_val_acc', 'final_val_acc']
display(comparison[main_columns].round(5))
display(comparison[['model', 'restored_epoch', 'final_val_loss',
                    'final_train_acc', 'train_val_gap']].round(5))
comparison.to_csv(OUT / 'model_comparison.csv', index=False)
best_name = comparison.sort_values(
    ['final_val_acc', 'final_val_loss', 'trainable_params'],
    ascending=[False, True, True]).iloc[0]['model']
print('Selected model:', best_name)
print('Accuracies above are fractions; multiply by 100 for percentages.')
print('A2 extra tuning fit time:', round(fold_results.time_s.sum(), 2), 'seconds')

with open(OUT / 'experiment_evidence.json', 'w') as f:
    json.dump(dict(seed=SEED, batch_size=BATCH_SIZE, tensorflow=tf.__version__,
                   records=records, histories=histories, best_a2=best_config,
                   selected_model=best_name, a2_tuning_fit_seconds=float(fold_results.time_s.sum())),
              f, indent=2)

best_val_pred = np.argmax(val_probabilities[best_name], axis=1)
cm = confusion_matrix(y_val, best_val_pred, labels=np.arange(10))
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    ax=ax, cmap='Blues', colorbar=False, xticks_rotation=45)
ax.set_title(f'{best_name}: final validation confusion matrix')
plt.tight_layout()
plt.savefig(OUT / 'confusion_matrix.png', dpi=160, bbox_inches='tight')
plt.show()
print(classification_report(y_val, best_val_pred, labels=np.arange(10),
                            target_names=class_names, zero_division=0))


In [ ]:
# Show 12 predictions; include up to 6 errors whenever errors are available.
wrong = np.flatnonzero(best_val_pred != y_val)
correct = np.flatnonzero(best_val_pred == y_val)
rng = np.random.default_rng(SEED)
wrong = rng.permutation(wrong)
correct = rng.permutation(correct)
n_wrong = min(6, len(wrong))
chosen = np.concatenate([wrong[:n_wrong], correct[:12 - n_wrong]])
if len(chosen) < 12:
    remaining = np.setdiff1d(np.arange(len(y_val)), chosen)
    chosen = np.concatenate([chosen, remaining[:12 - len(chosen)]])
print('Misclassified validation images:', len(wrong), '/', len(y_val))
if len(wrong) == 0:
    print('No misclassified examples exist in this run.')
fig, axes = plt.subplots(3, 4, figsize=(12, 8))
for ax, i in zip(axes.flat, chosen):
    truth, pred = int(y_val[i]), int(best_val_pred[i])
    ax.imshow(x_val[i].astype(np.uint8))
    ax.set_title(f'True: {class_names[truth]}\nPred: {class_names[pred]}',
                 color='green' if truth == pred else 'red', fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.savefig(OUT / 'validation_examples.png', dpi=160, bbox_inches='tight')
plt.show()


## 9. Predict the hidden test images and validate the submission
Reload the selected model's restored weights. Do not retrain on the validation set.
Map predictions to IDs from `test.npz`, then reorder to match `sample_submission.csv` exactly.


In [ ]:
reset_session()
if best_name == 'A1':
    best_model = build_a1()
elif best_name == 'A2':
    best_model = build_a2(best_config)
else:
    # All trained and frozen weights are restored next; no second ImageNet download is needed.
    best_model = build_a3(weights=None)
best_model.load_weights(str(OUT / f'{best_name}.weights.h5'))
test_probabilities = best_model.predict(x_test, batch_size=BATCH_SIZE, verbose=1)
assert test_probabilities.shape == (len(x_test), 10)
assert np.isfinite(test_probabilities).all()
predicted_labels = np.argmax(test_probabilities, axis=1).astype(np.int64)
by_id = pd.Series(predicted_labels, index=pd.Index(test_ids, name='id'))
submission = sample_submission[['id']].copy()
submission['label'] = submission['id'].map(by_id)
assert not submission.isna().any().any()
submission['label'] = submission['label'].astype(np.int64)
assert list(submission.columns) == ['id', 'label']
assert len(submission) == len(x_test) == len(sample_submission)
assert submission['id'].is_unique
assert submission['id'].equals(sample_submission['id'])
assert pd.api.types.is_integer_dtype(submission['label'])
assert submission['label'].between(0, 9).all()
submission.to_csv(OUT / 'submission.csv', index=False)
display(submission.head(12))
print('Saved:', OUT / 'submission.csv')
print('Verified row count, two column names, unique IDs, exact ID order, no missing values, and labels 0-9.')


## 10. Measured evidence for the discussion
The next cell prints observations from the actual run. Use them alongside the curves and
misclassifications to write your own explanation. A gap alone does not prove overfitting;
check whether training improves while validation loss stops improving or rises.


In [ ]:
for name in ['A1', 'A2', 'A3']:
    r = records[name]
    print(f"{name}: final validation {r['final_val_acc']:.2%}; "
          f"best epoch validation {r['best_val_acc']:.2%}; "
          f"restored epoch {r['restored_epoch']}; fit {r['time_s']:.1f}s; "
          f"training-validation gap {r['final_train_acc'] - r['final_val_acc']:.2%}.")
print(f"A2 selected {best_config['config']} using tuning CV only.")
print(f"Selected {best_name} with final validation accuracy {records[best_name]['final_val_acc']:.2%}.")
print('A1 first Dense parameters:', (32 * 32 * 3 + 1) * 256)
print('A3 trainable head parameters:', records['A3']['trainable_params'])

off_diagonal = cm.copy()
np.fill_diagonal(off_diagonal, 0)
if off_diagonal.max() > 0:
    actual, predicted = np.unravel_index(off_diagonal.argmax(), off_diagonal.shape)
    print('Most frequent directional confusion:', class_names[actual], '->',
          class_names[predicted], '| count:', off_diagonal[actual, predicted])


## 11. Student discussion: complete after execution
Replace the prompts below with your own observations before exporting.

1. **Grid choice:** Quote the selected configuration's mean and standard deviation,
   and compare it with the alternatives. Explain the stated tie-breaker if needed.
2. **Parameter efficiency:** Compare the measured totals. A Dense layer learns separate
   connections to flattened pixels; convolution shares small filters across image positions.
   Explain how this relates to your A1/A2 results. Do not claim fewer parameters without checking.
3. **Training behaviour:** Use the loss/accuracy curves, restored epochs and inference-mode
   train/validation gaps. Identify evidence consistent with underfitting or overfitting.
   Short training alone can leave a model undertrained; do not assume every gap is overfitting.
4. **Transfer learning:** A3 starts with ImageNet features, but only its head is trained.
   Explain the observed result, including low-resolution CIFAR images, resizing and domain difference.
   Frozen parameters still require forward computation, so fewer trainable parameters do not
   guarantee the shortest training time.
5. **Errors and conclusion:** Discuss two visible errors or the strongest confusion pair.
   Name the selected model and its measured validation accuracy. Avoid unsupported causal claims.

**Limitations:** Small 15,000-image labelled dataset; short 5/6/3-epoch budgets; one main split;
only two short tuning folds; selection and reporting share the same validation set;
unequal pretrained knowledge and resizing/forward-pass costs for A3; GPU timing variability;
light transformations in the hidden test set. These results do not establish a universal model ranking.

## 12. Submit exactly two graded PDFs
**Complete Notebook PDF:** Save an executed Kaggle version with every required output visible.
Restart and run all if time permits. Expand outputs, use browser Print / Save as PDF, and inspect
every page for missing plots or clipped code. Recommended name:
`PES1PG25CA053_UnitII_Notebook.pdf`.

**Supporting Experiment PDF (2-3 pages):** Write a concise report using actual results:
- Page 1: objective, dataset, fixed protocol, preprocessing and brief A1/A2/A3 descriptions.
- Page 2: four-row grid table, selection reasoning and final three-row comparison table.
- Page 3 if needed: evidence-based interpretation, limitations and conclusion.

Use `a2_grid_results.csv`, `model_comparison.csv`, curves and printed evidence as sources.
Do not copy the entire notebook into the report. Recommended name:
`PES1PG25CA053_UnitII_Report.pdf`.

The CSV and Kaggle leaderboard are formative feedback only. Upload the two PDFs to the
announced PESU Academy submission channel yourself.

## References
- Supplied DLIS Unit II Kaggle Competition handout, September 2026, pages 1-5.
- Supplied `DLIS_Unit_II_Starter_Notebook.ipynb`, workflow and prescribed settings.
- [TensorFlow ResNet50](https://www.tensorflow.org/api_docs/python/tf/keras/applications/ResNet50)
- [TensorFlow EarlyStopping](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping)
- [Keras transfer learning guide](https://keras.io/guides/transfer_learning/)

## Troubleshooting
- Missing files: attach the four competition files, then set `DATA_DIR` to their folder.
- Class CSV schema error: use the supplied file with `label` and `class_name` columns.
- Weight download failure: enable Internet or provide the official cached ImageNet weights.
  Do not replace the required pretrained run with random initialization.
- GPU memory error: the handout permits batch size 32. Set `BATCH_SIZE = 32`, restart and
  rerun all three approaches consistently; explain the change in the report.
- Slow search: keep the required 4,000 images, four configurations, two folds and two epochs.
